# ⚠️ SUPERSEDED — DO NOT RE-RUN THIS NOTEBOOK

**This notebook is superseded by `scripts/build_b2_temporal_course_stats.py` + `src/course_difficulty.py`.**

- It still contains the **old non-temporal difficulty logic**: a single pass that sees the
  whole train period at once when computing course statistics, instead of enriching each
  semester only from strictly earlier semesters (the leakage-safe B2 temporal rule).
- **It must not be re-run.** Its outputs are the same non-versioned
  `df_{train,valid,test}_difficulty.parquet` paths in `model_data/`, which have no version
  protection — re-running with the write step active would silently overwrite them with the
  old leaky version. As a safeguard, the save cell at the end now raises
  `RuntimeError('WRITE DISABLED: ...')` before any write; do not remove that guard.
- Use the versioned B2 builder instead: it writes only to `model_data/versions/<build_id>/`
  behind data gates and post-write read-back verification.

The content below is kept for reference only.

<!-- codex-architecture-notes -->
## Architectural Notes (legacy reference only)

**Purpose:** Builds empirical course-difficulty features from training data and enriches model splits.

**Notebook Shape:** 38 cells (14 code, 24 markdown).

**Inputs / Data Sources (base generation — D2):**
- `df_train = pd.read_parquet(df_train_base.parquet)`
- `df_valid = pd.read_parquet(df_valid_base.parquet)`
- `df_test  = pd.read_parquet(df_test_base.parquet)`

**Outputs / Side Effects (difficulty generation — D2, distinct files):**
- `df_train_enriched.to_parquet(df_train_difficulty.parquet, index=True)`
- `df_valid_enriched.to_parquet(df_valid_difficulty.parquet, index=True)`
- `df_test_enriched.to_parquet(df_test_difficulty.parquet,   index=True)`

**Logic Flow:**
1. Load base-generation train/valid/test model splits.
2. Compute course/degree historical difficulty statistics from training data.
3. Apply fallback levels for sparse courses.
4. Write the distinct difficulty-generation splits to disk (base files untouched).

**Maintainability Notes:** Difficulty features must be computed from training history only; keep leakage checks explicit when refactoring this into source code.

# Course Difficulty Features

**Step A** — Load `df_train`, `df_valid`, `df_test` from the pre-built parquet files saved by `01_split_diagnostics.ipynb`. Run that notebook first if the files don't exist.

**Step B** — Build per-course difficulty features from `df_train` only (empirical Bayes shrinkage, 3-level fallback chain, leave-one-out), join to all three splits, and save.

The split boundaries and exclusion logic live entirely in `01_split_diagnostics.ipynb`. This notebook never touches `df_primary` or redefines any year/semester masks.

## Step A — Load Pre-built Splits

The temporal split was defined and saved by `01_split_diagnostics.ipynb`. This notebook loads the three parquet files directly — it never redefines year/semester boundaries or touches `df_primary`.

If any sanity check below fails, re-run `01_split_diagnostics.ipynb` first.

**Step A.0 — Configuration.** `SPLIT_DATA_DIR` must match the same constant in `01_split_diagnostics.ipynb`. Edit this path here if the split files have moved.

In [1]:
from src.paths import MODEL_DATA_DIR, assert_data_root
from src.io_utils import save_parquet
SPLIT_DATA_DIR = str(MODEL_DATA_DIR)

**Step A.1** — Load `df_train`, `df_valid`, `df_test` from the saved parquet files. Print each shape immediately to confirm the files loaded correctly.

In [2]:
import pandas as pd
import numpy as np
import os

# D2 (distinct split generations): read the BASE generation written by
# 01_split_diagnostics.ipynb; this notebook writes its own DIFFICULTY generation.
TRAIN_PATH = os.path.join(SPLIT_DATA_DIR, 'df_train_base.parquet')
VALID_PATH = os.path.join(SPLIT_DATA_DIR, 'df_valid_base.parquet')
TEST_PATH  = os.path.join(SPLIT_DATA_DIR, 'df_test_base.parquet')

# Data-root guard (governance contract 12): fail loudly if the resolved root is
# a freshly created empty tree or the base-generation inputs are absent.
assert_data_root(TRAIN_PATH, VALID_PATH, TEST_PATH)

df_train = pd.read_parquet(TRAIN_PATH)
df_valid = pd.read_parquet(VALID_PATH)
df_test  = pd.read_parquet(TEST_PATH)

print(f'df_train : {df_train.shape}')
print(f'df_valid : {df_valid.shape}')
print(f'df_test  : {df_test.shape}')

df_train : (450465, 63)
df_valid : (156097, 63)
df_test  : (110008, 63)


In [3]:

# ── Step A.1b: Snapshot old fallback levels and remove stale difficulty columns ──
#
# The parquet files on disk may carry columns from a PREVIOUS run of this notebook.
# Before re-enriching, we must:
#   1. Capture the old per-row fallback level for valid/test — used later in
#      B.13 to print old → new transition tables.
#   2. Drop all old difficulty columns so the column count returns to the raw
#      base. This prevents duplicate columns on re-run.
#
# We drop all known difficulty column names (any past scheme), making this
# cell idempotent: safe to run multiple times.

OLD_DIFF_COLS = [
    # 3-level scheme (original version of this notebook)
    'course_pass_rate',
    'course_avg_mark',
    'course_retake_rate',
    'course_difficulty_fallback_level',
    'support_count',
    # 5-level scheme (previous version)
    'course_pass_rate_historical',
    'course_avg_mark_historical',
    'course_retake_rate_historical',
    'course_history_count',
    'difficulty_fallback_level',
    'course_difficulty_missing',
    # 6-level scheme (current version — dropped so we start clean on re-run)
    'difficulty_group_support_count',
]

# ── 1. Snapshot old fallback level for transition-table analysis ──
# Try both possible column names from past schemes.
_old_col_valid = None
_old_col_test  = None

for col in ['course_difficulty_fallback_level', 'difficulty_fallback_level']:
    if col in df_valid.columns and _old_col_valid is None:
        _old_col_valid = df_valid[col].copy()
    if col in df_test.columns and _old_col_test is None:
        _old_col_test = df_test[col].copy()

if _old_col_valid is not None:
    print(f'Saved old valid fallback levels — values: {sorted(_old_col_valid.unique())}')
else:
    print('No old fallback-level column in df_valid (first run — no transition table available).')

if _old_col_test is not None:
    print(f'Saved old test  fallback levels — values: {sorted(_old_col_test.unique())}')
else:
    print('No old fallback-level column in df_test  (first run — no transition table available).')

# ── 2. Drop all old difficulty columns from all three splits (in-place) ──
for df in [df_train, df_valid, df_test]:
    cols_to_drop = []
    for c in OLD_DIFF_COLS:
        if c in df.columns:
            cols_to_drop.append(c)
    if cols_to_drop:
        df.drop(columns=cols_to_drop, inplace=True)

print(f'\nBase column count after removing old difficulty columns:')
print(f'  df_train : {df_train.shape[1]}')
print(f'  df_valid : {df_valid.shape[1]}')
print(f'  df_test  : {df_test.shape[1]}')
print('(All three should match — any mismatch means duplicate removal failed.)')

assert df_train.shape[1] == df_valid.shape[1] == df_test.shape[1], (
    'FAIL: Base column counts differ across splits after dropping old columns.'
)


No old fallback-level column in df_valid (first run — no transition table available).
No old fallback-level column in df_test  (first run — no transition table available).

Base column count after removing old difficulty columns:
  df_train : 63
  df_valid : 63
  df_test  : 63
(All three should match — any mismatch means duplicate removal failed.)


In [4]:

# ── Step A.1c: Assert faculty_id exists in all three splits ──
#
# faculty_id is used as Level-4 fallback key (faculty+req_type+credits).
# It must already be present in the parquet files — loaded from df_primary
# by 01_split_diagnostics.ipynb.
#
# DO NOT re-join faculty_id from any source table here. A re-join would
# risk duplicating rows, reordering the index, or introducing a different
# encoding than what split_diagnostics used. If faculty_id is missing,
# the split files are stale — re-run 01_split_diagnostics.ipynb.

for _split_name, _df_split in [('train', df_train), ('valid', df_valid), ('test', df_test)]:
    if 'faculty_id' not in _df_split.columns:
        raise AssertionError(
            f'\nFAIL: faculty_id is missing from the {_split_name} split.\n'
            f'The parquet files were built without faculty_id in df_primary.\n'
            f'Re-run 01_split_diagnostics.ipynb to include faculty_id, then reload here.\n'
            f'DO NOT fix this by merging from a source table — that risks row reordering.'
        )

print('faculty_id present in all three splits: OK')
print()
print('faculty_id null percentage per split (computed from the loaded splits):')
for _split_name, _df_split in [('train', df_train), ('valid', df_valid), ('test', df_test)]:
    _n_total = len(_df_split)
    _n_null  = int(_df_split['faculty_id'].isna().sum())
    _pct     = _n_null / _n_total * 100
    print(f'  {_split_name:<6}: {_n_null:>7,} / {_n_total:>7,} null  ({_pct:.4f}%)')


faculty_id present in all three splits: OK

faculty_id null percentage per split (computed from the loaded splits):
  train :       0 / 450,465 null  (0.0000%)
  valid :       0 / 156,097 null  (0.0000%)
  test  :       0 / 110,008 null  (0.0000%)


**What this shows:** Shapes confirm all three files loaded. If a file is missing, `pd.read_parquet` raises `FileNotFoundError` — re-run `01_split_diagnostics.ipynb` first to build the base generation (`df_*_base.parquet`). The column count here is the **before-enrichment** baseline (after Step A.1b drops any stale difficulty columns); the save cell at the end will assert it increased by exactly the expected difficulty columns and write the distinct `df_*_difficulty.parquet` files.

**Step A.2** — Confirm the loaded splits have the expected year boundaries (train 2005–2021, val 2022–2023, test includes up to 2025-semester-1).

In [5]:
rows = []
for name, df in [('train', df_train), ('valid', df_valid), ('test', df_test)]:
    n = len(df)
    rows.append({
        'split':     name,
        'row_count': n,
        'min_year':  df['part_year'].min(),
        'max_year':  df['part_year'].max(),
    })
print(pd.DataFrame(rows).to_string(index=False))

split  row_count  min_year  max_year
train     450465      2005      2021
valid     156097      2022      2023
 test     110008      2024      2025


**What this shows:** Min/max year per split confirms the correct temporal boundaries. If `min_year` for train is 2013 instead of 2005, the loaded files are stale — re-run `01_split_diagnostics.ipynb` with the corrected boundaries first. This cell does **not** re-derive the split; it only confirms what's already in the files.

**Step A.3** — Hard sanity checks: 2016-semester-4 rows must be in `df_train` (kept intentionally), and 2025-semester-2 rows must be absent from all three splits (excluded as incomplete). Raises a clear `AssertionError` if either check fails.

In [6]:
# Check 1: 2016-semester-4 must be present in train (intentionally kept)
year_f_tr = df_train['part_year'].astype('float64')
sem_f_tr  = df_train['part_semester'].astype('float64')
n_2016_s4 = int(((year_f_tr == 2016) & (sem_f_tr == 4)).sum())

# Check 2: 2025-semester-2 must be absent from all three splits (excluded as incomplete)
n_2025_s2 = sum(
    int(((df['part_year'].astype('float64') == 2025) &
         (df['part_semester'].astype('float64') == 2)).sum())
    for df in [df_train, df_valid, df_test]
)

print(f'df_train rows with part_year==2016 AND part_semester==4 : {n_2016_s4:,}')
print(f'Rows with part_year==2025 AND part_semester==2 (all splits) : {n_2025_s2:,}')

assert n_2016_s4 > 0, (
    'FAIL: No 2016-semester-4 rows in df_train. '
    'The split files are stale or were built with the wrong train boundary. '
    'Re-run 01_split_diagnostics.ipynb first.'
)
assert n_2025_s2 == 0, (
    f'FAIL: Found {n_2025_s2:,} rows with part_year==2025 AND part_semester==2 in the splits. '
    'The incomplete semester should have been excluded. '
    'Re-run 01_split_diagnostics.ipynb first.'
)

print('\nSanity checks PASSED.')
print('  -> 2016-semester-4 rows are present in train (kept intentionally).')
print('  -> 2025-semester-2 rows are absent from all splits (excluded as incomplete).')

df_train rows with part_year==2016 AND part_semester==4 : 5,166
Rows with part_year==2025 AND part_semester==2 (all splits) : 0

Sanity checks PASSED.
  -> 2016-semester-4 rows are present in train (kept intentionally).
  -> 2025-semester-2 rows are absent from all splits (excluded as incomplete).


**What this shows:** Two hard checks that the loaded files were built with the correct logic. If either assertion fires, the error message tells you exactly what's wrong and instructs you to re-run `01_split_diagnostics.ipynb` — this notebook will not silently continue on stale or incorrect split files.

---

## Step B — Course Difficulty Features (train-only aggregation)

All aggregations in this step use **`df_train` exclusively**. `df_valid` and `df_test` are never touched until the join in B.13.

Key design decisions:
- **Empirical Bayes shrinkage** (`MIN_SUPPORT = 20`) prevents low-support courses from carrying spurious extremes.
- **6-level fallback chain** handles cold-start cases: same course in a new degree (Level 2), new courses with degree-level structural signal (Level 3), faculty-level structural signal (Level 4), university-wide signal (Level 5), and pure cold-start (Level 6 / global).
- **Leave-one-out for train rows** (Level 1 only) prevents a row's own grade from leaking into its own feature value.
- **`course_difficulty_missing` flag** marks every row that did not receive a confident Level-1 estimate with adequate support.
- **`course_history_count`** carries REAL course-level history only (Levels 1–2); set to 0 for Levels 3–6 to avoid falsely inflating confidence.
- **`difficulty_group_support_count`** is an audit-only column: the raw support of whichever fallback group fired at any level.


**Step B.9** — Aggregate per `degree_course_key` from `df_train`: support count, pass rate, average mark, retake rate.

In [7]:
# All aggregations are from df_train ONLY — val and test never touched here
# NaN final_mark / attempt_number values are excluded from each lambda via .dropna()

key_stats = (
    df_train
    .groupby('degree_course_key', as_index=False)
    .agg(
        support_count=('final_mark', 'count'),
        course_pass_rate=(
            'final_mark',
            lambda x: (x.dropna() >= 50).mean() if x.notna().any() else float('nan')
        ),
        course_avg_mark=('final_mark', 'mean'),
        course_retake_rate=(
            'attempt_number',
            lambda x: (x.dropna() > 1).mean() if x.notna().any() else float('nan')
        ),
    )
)

print(f'Unique degree_course_key in train : {len(key_stats):,}')
print(f'\nsupport_count distribution:')
print(key_stats['support_count'].describe().to_string())
print(f'\nKeys with support_count < 20  : {(key_stats["support_count"] < 20).sum():,}')
print(f'Keys with support_count == 1  : {(key_stats["support_count"] == 1).sum():,}')
print(f'\nFirst 5 rows:')
print(key_stats.head(5).to_string(index=False))

Unique degree_course_key in train : 1,666

support_count distribution:
count        1666.0
mean     270.387155
std      527.026279
min             1.0
25%             5.0
50%            53.0
75%          212.75
max          3180.0

Keys with support_count < 20  : 601
Keys with support_count == 1  : 141

First 5 rows:
degree_course_key  support_count  course_pass_rate  course_avg_mark  course_retake_rate
  1.111__1015.111            192           0.90625        76.119792            0.067708
  1.111__1016.111            914          0.974836        79.818381            0.040481
  1.111__1017.111            299          0.926421        77.414716            0.086957
  1.111__1018.111            524          0.885496        66.753817            0.158397
  1.111__1019.111            938          0.934968        78.590618            0.085288


**What this shows:** How many distinct `degree_course_key` values exist in train, and how unevenly they're seen. A large number of keys with `support_count < 20` means shrinkage (B.10) will have a meaningful effect on a significant fraction of keys. Keys with `support_count == 1` will collapse almost entirely to the global average after shrinkage.

**Step B.10** — Apply empirical Bayes shrinkage to `course_pass_rate` and `course_avg_mark`. Formula: `(n * raw + MIN_SUPPORT * global) / (n + MIN_SUPPORT)`. Print a before/after comparison for 10 low-support and 10 high-support keys.

In [8]:
MIN_SUPPORT = 20  # course with exactly this many observations is blended 50/50 with global avg

global_pass_rate = (df_train['final_mark'].dropna() >= 50).mean()
global_avg_mark  = df_train['final_mark'].mean()

print(f'MIN_SUPPORT      : {MIN_SUPPORT}')
print(f'Global pass rate : {global_pass_rate:.4f}')
print(f'Global avg mark  : {global_avg_mark:.4f}')

# Preserve raw values for the before/after printout for shrinking 
key_stats['course_pass_rate_raw'] = key_stats['course_pass_rate'].copy()
key_stats['course_avg_mark_raw']  = key_stats['course_avg_mark'].copy()

# Shrinkage: pull toward global mean weighted by how little we've seen the course
key_stats['course_pass_rate'] = (
    (key_stats['support_count'] * key_stats['course_pass_rate'] + MIN_SUPPORT * global_pass_rate)
    / (key_stats['support_count'] + MIN_SUPPORT)
)
key_stats['course_avg_mark'] = (
    (key_stats['support_count'] * key_stats['course_avg_mark'] + MIN_SUPPORT * global_avg_mark)
    / (key_stats['support_count'] + MIN_SUPPORT)
)

show_cols = ['degree_course_key', 'support_count',
             'course_pass_rate_raw', 'course_pass_rate',
             'course_avg_mark_raw',  'course_avg_mark']

pd.set_option('display.float_format', '{:.4f}'.format)

print(f'\n--- 10 LOWEST-support courses (shrinkage pulls strongly toward global) ---')
print(key_stats.nsmallest(10, 'support_count')[show_cols].to_string(index=False))

print(f'\n--- 10 HIGHEST-support courses (shrinkage has minimal effect) ---')
print(key_stats.nlargest(10, 'support_count')[show_cols].to_string(index=False))

pd.reset_option('display.float_format')

MIN_SUPPORT      : 20
Global pass rate : 0.8413
Global avg mark  : 65.5821

--- 10 LOWEST-support courses (shrinkage pulls strongly toward global) ---
degree_course_key  support_count  course_pass_rate_raw  course_pass_rate  course_avg_mark_raw  course_avg_mark
   1.111__211.111              1                0.0000            0.8013              17.0000          63.2687
   1.111__223.111              1                0.0000            0.8013              47.0000          64.6972
   1.111__224.111              1                0.0000            0.8013              30.0000          63.8877
   1.111__229.111              1                0.0000            0.8013              46.0000          64.6496
   1.111__230.111              1                1.0000            0.8489              60.0000          65.3163
   1.111__234.111              1                0.0000            0.8013              35.0000          64.1258
   1.111__237.111              1                0.0000            0.8013

**What this shows:** The shrinkage effect in action. Low-support courses (e.g., seen once) should show `course_pass_rate` much closer to the global baseline than `course_pass_rate_raw`. High-support courses should show very little movement. `course_retake_rate` is left unshrunk — it will still appear in the final output from B.9.

**Step B.11** — Build Levels 2–5 fallback structures and the global (Level 6) scalar. All aggregations use `df_train` only.

- **Level 2** (`course_id`): same course under any degree — covers courses that appear in valid/test under a new degree not seen in train.
- **Level 3** (`degree_id + requirement_type_id + course_credits_key`): similar courses in the same program. Keys built only when all three fields are non-null.
- **Level 4** (`faculty_id + requirement_type_id + course_credits_key`): similar courses in the same faculty. Broader than Level 3 (a faculty spans several degrees). Keys built only when `faculty_id`, `requirement_type_id`, AND `course_credits` are all non-null — missing `faculty_id` falls through to Level 5.
- **Level 5** (`requirement_type_id + course_credits_key`): similar courses across the whole university. Keys built only when `requirement_type_id` AND `course_credits` are non-null.
- **Level 6**: global average scalar — last resort, no key needed.

Shrinkage (`MIN_SUPPORT = 20`) is applied to `course_pass_rate` and `course_avg_mark` at Levels 2–5. This prevents bad keys like `"nan__3__2"` — the null-mask is checked before any string concatenation.


In [9]:

# ── Step B.11: Build Level-2 through Level-5 fallback structures + global (Level 6) ──
#
# All aggregations are computed from df_train ONLY.
# df_valid and df_test are never touched in this cell.
#
# Level 1 stats (key_stats, non-LOO) were built in B.9/B.10 above.
# This cell adds:
#   Level 2 → course_id across all degrees (same course, different degree pairing)
#   Level 3 → degree_id + requirement_type_id + course_credits_key (same program, similar course)
#   Level 4 → faculty_id + requirement_type_id + course_credits_key (same faculty, similar course) ← NEW
#   Level 5 → requirement_type_id + course_credits_key (whole university, similar course)
#   Level 6 → global average (last resort)

# Add course_id to key_stats for audit use in B.14 
key_stats['course_id'] = key_stats['degree_course_key'].str.rsplit('__', n=1).str[-1]  

# Pre-compute global retake rate here so it is available for Level 6 and shrinkage below
global_retake_rate    = (df_train['attempt_number'].dropna() > 1).mean()
# Pre-compute global support count for the difficulty_group_support_count audit column at Level 6
global_support_count  = int(df_train['final_mark'].count())

print(f'Global retake rate : {global_retake_rate:.4f}')
print(f'Global support count (final_mark not-null rows in train): {global_support_count:,}')


# ════════════════════════════════════════════════════════
# Level 2 — course_id alone, cross-degree
# ════════════════════════════════════════════════════════
# Covers rows where the course exists in train under some degree but
# valid/test see it under a NEW degree not in train.

_cid_col = df_train['degree_course_key'].str.rsplit('__', n=1).str[-1]

cid_stats = (
    df_train
    .assign(course_id=_cid_col.values)
    .groupby('course_id', as_index=False)
    .agg(
        support_count=('final_mark', 'count'),
        course_pass_rate=(
            'final_mark',
            lambda x: (x.dropna() >= 50).mean() if x.notna().any() else float('nan')
        ),
        course_avg_mark=('final_mark', 'mean'),
        course_retake_rate=(
            'attempt_number',
            lambda x: (x.dropna() > 1).mean() if x.notna().any() else float('nan')
        ),
    )
)

cid_stats['course_pass_rate'] = (
    (cid_stats['support_count'] * cid_stats['course_pass_rate'] + MIN_SUPPORT * global_pass_rate)
    / (cid_stats['support_count'] + MIN_SUPPORT)
)
cid_stats['course_avg_mark'] = (
    (cid_stats['support_count'] * cid_stats['course_avg_mark'] + MIN_SUPPORT * global_avg_mark)
    / (cid_stats['support_count'] + MIN_SUPPORT)
)

print(f'\nLevel-2 (course_id alone)                   unique keys : {len(cid_stats):,}')


# ════════════════════════════════════════════════════════
# Level 3 — degree_id + requirement_type_id + course_credits_key
# ════════════════════════════════════════════════════════
# Similar courses inside the same degree/program.
# Designed for genuinely new course_ids that still belong to a known degree
# with a recognisable requirement type and credit count.
#
# Key is built ONLY when all three fields are non-null.
# This prevents bad keys like "nan__3__2".

_l3_mask = (
    df_train['degree_id'].notna()
    & df_train['requirement_type_id'].notna()
    & df_train['course_credits'].notna()
)

_l3_df = df_train.loc[_l3_mask].copy()
_l3_df['course_credits_key'] = _l3_df['course_credits'].astype('float64').round().astype(int)
_l3_df['l3_key'] = (
    _l3_df['degree_id'].astype(str)
    + '__'
    + _l3_df['requirement_type_id'].astype(str)
    + '__'
    + _l3_df['course_credits_key'].astype(str)
)

l3_stats = (
    _l3_df
    .groupby('l3_key', as_index=False)
    .agg(
        support_count=('final_mark', 'count'),
        course_pass_rate=(
            'final_mark',
            lambda x: (x.dropna() >= 50).mean() if x.notna().any() else float('nan')
        ),
        course_avg_mark=('final_mark', 'mean'),
        course_retake_rate=(
            'attempt_number',
            lambda x: (x.dropna() > 1).mean() if x.notna().any() else float('nan')
        ),
    )
)

l3_stats['course_pass_rate'] = (
    (l3_stats['support_count'] * l3_stats['course_pass_rate'] + MIN_SUPPORT * global_pass_rate)
    / (l3_stats['support_count'] + MIN_SUPPORT)
)
l3_stats['course_avg_mark'] = (
    (l3_stats['support_count'] * l3_stats['course_avg_mark'] + MIN_SUPPORT * global_avg_mark)
    / (l3_stats['support_count'] + MIN_SUPPORT)
)

print(f'Level-3 (degree+req_type+credits)           unique keys : {len(l3_stats):,}')
print(f'  (train rows excluded due to missing fields: {(~_l3_mask).sum():,})')


# ════════════════════════════════════════════════════════
# Level 4 — faculty_id + requirement_type_id + course_credits_key   ← NEW
# ════════════════════════════════════════════════════════
# Similar courses inside the same FACULTY.
# Broader than degree-level (Level 3) but tighter than university-wide (Level 5).
# A faculty groups several related degree programs — if the same faculty offers
# courses with the same requirement type and credit count, they are a reasonable
# prior for a new course in that faculty.
#
# Key is built ONLY when faculty_id, requirement_type_id, AND course_credits
# are all non-null. If faculty_id is missing, this level is skipped.

_l4_mask = (
    df_train['faculty_id'].notna()
    & df_train['requirement_type_id'].notna()
    & df_train['course_credits'].notna()
)

_l4_df = df_train.loc[_l4_mask].copy()
_l4_df['course_credits_key'] = _l4_df['course_credits'].astype('float64').round().astype(int)

# Key format: "faculty_id__requirement_type_id__credits_key"
_l4_df['l4_key'] = (
    _l4_df['faculty_id'].astype(str)
    + '__'
    + _l4_df['requirement_type_id'].astype(str)
    + '__'
    + _l4_df['course_credits_key'].astype(str)
)

l4_stats = (
    _l4_df
    .groupby('l4_key', as_index=False)
    .agg(
        support_count=('final_mark', 'count'),
        course_pass_rate=(
            'final_mark',
            lambda x: (x.dropna() >= 50).mean() if x.notna().any() else float('nan')
        ),
        course_avg_mark=('final_mark', 'mean'),
        course_retake_rate=(
            'attempt_number',
            lambda x: (x.dropna() > 1).mean() if x.notna().any() else float('nan')
        ),
    )
)

l4_stats['course_pass_rate'] = (
    (l4_stats['support_count'] * l4_stats['course_pass_rate'] + MIN_SUPPORT * global_pass_rate)
    / (l4_stats['support_count'] + MIN_SUPPORT)
)
l4_stats['course_avg_mark'] = (
    (l4_stats['support_count'] * l4_stats['course_avg_mark'] + MIN_SUPPORT * global_avg_mark)
    / (l4_stats['support_count'] + MIN_SUPPORT)
)

print(f'Level-4 (faculty+req_type+credits)          unique keys : {len(l4_stats):,}')
print(f'  (train rows excluded due to missing faculty/fields: {(~_l4_mask).sum():,})')


# ════════════════════════════════════════════════════════
# Level 5 — requirement_type_id + course_credits_key (university-wide)
# ════════════════════════════════════════════════════════
# Similar courses pooled across the WHOLE UNIVERSITY.
# No degree or faculty constraint — broadest grouping before global fallback.
#
# Key is built ONLY when requirement_type_id AND course_credits are non-null.

_l5_mask = (
    df_train['requirement_type_id'].notna()
    & df_train['course_credits'].notna()
)

_l5_df = df_train.loc[_l5_mask].copy()
_l5_df['course_credits_key'] = _l5_df['course_credits'].astype('float64').round().astype(int)

# Key format: "requirement_type_id__credits_key"
_l5_df['l5_key'] = (
    _l5_df['requirement_type_id'].astype(str)
    + '__'
    + _l5_df['course_credits_key'].astype(str)
)

l5_stats = (
    _l5_df
    .groupby('l5_key', as_index=False)
    .agg(
        support_count=('final_mark', 'count'),
        course_pass_rate=(
            'final_mark',
            lambda x: (x.dropna() >= 50).mean() if x.notna().any() else float('nan')
        ),
        course_avg_mark=('final_mark', 'mean'),
        course_retake_rate=(
            'attempt_number',
            lambda x: (x.dropna() > 1).mean() if x.notna().any() else float('nan')
        ),
    )
)

l5_stats['course_pass_rate'] = (
    (l5_stats['support_count'] * l5_stats['course_pass_rate'] + MIN_SUPPORT * global_pass_rate)
    / (l5_stats['support_count'] + MIN_SUPPORT)
)
l5_stats['course_avg_mark'] = (
    (l5_stats['support_count'] * l5_stats['course_avg_mark'] + MIN_SUPPORT * global_avg_mark)
    / (l5_stats['support_count'] + MIN_SUPPORT)
)

print(f'Level-5 (req_type+credits, univ-wide)       unique keys : {len(l5_stats):,}')


# ════════════════════════════════════════════════════════
# Level 6 — global average (last resort)
# ════════════════════════════════════════════════════════
# Used only when all five specific levels fail to match.

global_vals = {
    'course_pass_rate'  : global_pass_rate,
    'course_avg_mark'   : global_avg_mark,
    'course_retake_rate': global_retake_rate,
}

print(f'\nLevel-6 global averages (last resort):')
for k, v in global_vals.items():
    print(f'  {k:<28}: {v:.4f}')
print(f'  global_support_count             : {global_support_count:,}')


Global retake rate : 0.1606
Global support count (final_mark not-null rows in train): 450,465

Level-2 (course_id alone)                   unique keys : 811
Level-3 (degree+req_type+credits)           unique keys : 197
  (train rows excluded due to missing fields: 0)
Level-4 (faculty+req_type+credits)          unique keys : 83
  (train rows excluded due to missing faculty/fields: 0)
Level-5 (req_type+credits, univ-wide)       unique keys : 24

Level-6 global averages (last resort):
  course_pass_rate            : 0.8413
  course_avg_mark             : 65.5821
  course_retake_rate          : 0.1606
  global_support_count             : 450,465


**What this shows:** Key counts for each fallback level confirm the hierarchy. Level 2 keys ≤ Level 1 keys (many `degree_course_key` values collapse to the same `course_id`). Level 3 is scoped to individual degrees so has more keys than Level 4 (faculty) or Level 5 (university-wide). Level 4 keys are bounded by the number of unique (faculty, req_type, credits) combinations seen in train — expect more keys than Level 5 but fewer than Level 3. Rows excluded from Level 3/4/5 aggregation (missing required fields) are shown explicitly and will fall through at inference time.


**Step B.12** — Leave-one-out (LOO) for train rows. Each train row's own outcome is subtracted from its course's running totals before shrinkage is applied, preventing self-leakage.

In [10]:

# ── Step B.12: Leave-one-out (LOO) enrichment for df_train ──
#
# LOO stays on the Level-1 train path ONLY.
# Every train row's degree_course_key exists in train by construction,
# so ALL train rows are Level 1. Levels 2–6 apply to valid/test only.
#
# Why LOO: without it, each row's own grade would enter its course's aggregate
# and leak into its own feature value (data snooping).
# LOO subtracts each row's own contribution before computing the shrunken stat.

# ── Step 1: per-key totals from df_train ──
key_totals = (
    df_train
    .groupby('degree_course_key')
    .agg(
        n_fm    =('final_mark',     'count'),
        sum_pass=('final_mark',     lambda x: int((x.dropna() >= 50).sum())),
        sum_mark=('final_mark',     lambda x: float(x.dropna().sum())),
        n_at    =('attempt_number', 'count'),
        sum_ret =('attempt_number', lambda x: int((x.dropna() > 1).sum())),
    )
    .reset_index()
)

# ── Step 2: map per-key totals onto every train row (preserves original index) ──
_idx = key_totals.set_index('degree_course_key')
df_train_loo = df_train.copy()

for col in ['n_fm', 'sum_pass', 'sum_mark', 'n_at', 'sum_ret']:
    df_train_loo[col] = df_train_loo['degree_course_key'].map(_idx[col])

# ── Step 3: this row's own contribution to the aggregate ──
_fm_ok     = df_train_loo['final_mark'].notna()
_this_pass = ((df_train_loo['final_mark'] >= 50) & _fm_ok).astype(int)
_this_mark = df_train_loo['final_mark'].fillna(0.0)
_this_fm_c = _fm_ok.astype(int)

_at_ok     = df_train_loo['attempt_number'].notna()
_this_ret  = ((df_train_loo['attempt_number'] > 1) & _at_ok).astype(int)
_this_at_c = _at_ok.astype(int)

# ── Step 4: LOO sums — subtract this row's own contribution ──
_n_loo        = (df_train_loo['n_fm']    - _this_fm_c).clip(lower=0)
_sum_pass_loo = (df_train_loo['sum_pass']- _this_pass).clip(lower=0)
_sum_mark_loo = (df_train_loo['sum_mark']- _this_mark)
_n_ret_loo    = (df_train_loo['n_at']    - _this_at_c).clip(lower=0)
_sum_ret_loo  = (df_train_loo['sum_ret'] - _this_ret).clip(lower=0)

# ── Step 5: shrunk LOO pass_rate and avg_mark ──
# Singleton rows (n_loo == 0): denominator collapses to MIN_SUPPORT → global average.
df_train_loo['course_pass_rate_historical'] = (
    (_sum_pass_loo + MIN_SUPPORT * global_pass_rate) / (_n_loo + MIN_SUPPORT)
)
df_train_loo['course_avg_mark_historical'] = (
    (_sum_mark_loo + MIN_SUPPORT * global_avg_mark) / (_n_loo + MIN_SUPPORT)
)

# ── Step 6: LOO retake rate — global fallback for singleton rows ──
df_train_loo['course_retake_rate_historical'] = np.where(
    _n_ret_loo.values > 0,
    (_sum_ret_loo / _n_ret_loo.replace(0, np.nan)).values,
    global_retake_rate
)

# ── Step 7: history count, group support count, and fallback level ──
#
# course_history_count: REAL course-level history only.
#   For Level 1 train rows this equals n_loo (the LOO support count for
#   this specific degree_course_key). Level 1 is the only level where
#   course_history_count carries real course history.
#
# difficulty_group_support_count: raw support of whichever fallback group fired.
#   For Level 1, the group IS the course, so this equals course_history_count.
#   For Levels 2–6 (valid/test only), it will differ — set in B.13.

df_train_loo['course_history_count']          = _n_loo.astype(int)
df_train_loo['difficulty_group_support_count'] = _n_loo.astype(int)  # group == course at Level 1
df_train_loo['difficulty_fallback_level']      = 1

# ── Step 8: course_difficulty_missing flag ──
# Rule: 0 (confident) ONLY when fallback level == 1 AND history count >= MIN_SUPPORT.
# Singletons and all non-Level-1 rows are flagged missing = 1.
df_train_loo['course_difficulty_missing'] = (
    (df_train_loo['difficulty_fallback_level'] != 1)
    | (df_train_loo['course_history_count'] < MIN_SUPPORT)
).astype(int)

# Drop the intermediate aggregation columns
_drop_cols = ['n_fm', 'sum_pass', 'sum_mark', 'n_at', 'sum_ret']
df_train_enriched = df_train_loo.drop(columns=_drop_cols)

n_singleton = int((_n_loo == 0).sum())
print(f'LOO enrichment complete: {len(df_train_enriched):,} train rows')
print(f'Singleton rows (n_loo == 0, collapse to global avg via shrinkage): {n_singleton:,}')

print(f'\nTrain course_difficulty_missing distribution:')
cdm_counts = df_train_enriched['course_difficulty_missing'].value_counts().sort_index()
for val, cnt in cdm_counts.items():
    print(f'  missing={val}: {cnt:>8,} rows  ({cnt / len(df_train_enriched) * 100:.2f}%)')

print(f'\nSample — 5 unique keys:')
_show_cols = [
    'degree_course_key', 'course_history_count', 'difficulty_group_support_count',
    'course_pass_rate_historical', 'course_avg_mark_historical',
    'difficulty_fallback_level', 'course_difficulty_missing',
]
print(
    df_train_enriched[_show_cols]
    .drop_duplicates('degree_course_key')
    .head(5)
    .to_string(index=False)
)


LOO enrichment complete: 450,465 train rows
Singleton rows (n_loo == 0, collapse to global avg via shrinkage): 141

Train course_difficulty_missing distribution:
  missing=0:  447,642 rows  (99.37%)
  missing=1:    2,823 rows  (0.63%)

Sample — 5 unique keys:
degree_course_key  course_history_count  difficulty_group_support_count  course_pass_rate_historical  course_avg_mark_historical  difficulty_fallback_level  course_difficulty_missing
  3.111__1016.111                   147                             147                     0.951056                   72.243365                          1                          0
  3.111__1017.111                    76                              76                     0.904441                    71.84002                          1                          0
  3.111__1019.111                   292                             292                     0.912905                   72.800134                          1                          0
   3.111

**What this shows:** `course_history_count` for train rows is `n_loo` (the course's total count minus 1 for this row), not the full count — this is expected. Singleton rows (history count = 0) receive the global average via shrinkage, which is the correct conservative behaviour. All train rows get `difficulty_fallback_level = 1` by construction, and the `course_difficulty_missing` flag captures which of them still have insufficient support (history count < MIN_SUPPORT).

**Step B.13** — Apply the 6-level fallback chain to `df_valid` and `df_test`. Print the fallback distribution for all three splits.

Fallback hierarchy:
- **Level 1**: `degree_course_key` — exact same course in the same degree (exact hit in train). `course_history_count` = LOO support count (real history). `difficulty_group_support_count` = same.
- **Level 2**: `course_id` — same course appearing under a new degree not seen in train. `course_history_count` = course_id support count (real history). `difficulty_group_support_count` = same.
- **Level 3**: `degree_id + requirement_type_id + course_credits_key` — new course_id, same program, similar type and credit count. `course_history_count` = **0** (not real history for this course). `difficulty_group_support_count` = L3 group support.
- **Level 4**: `faculty_id + requirement_type_id + course_credits_key` — new course_id, same faculty, similar type and credit count. `course_history_count` = **0**. `difficulty_group_support_count` = L4 group support. Level 4 requires `faculty_id` to be non-null; missing `faculty_id` falls to Level 5.
- **Level 5**: `requirement_type_id + course_credits_key` — similar type and credit count across the whole university. `course_history_count` = **0**. `difficulty_group_support_count` = L5 group support.
- **Level 6**: Global average — pure cold-start, no structural signal available. `course_history_count` = **0**. `difficulty_group_support_count` = total train observations.

`course_difficulty_missing = 0` **only** for Level 1 with `course_history_count >= MIN_SUPPORT`. All other levels (including Level 2, which has real course history) are flagged `missing = 1` — Level 2 is still uncertain because the degree pairing is new.

Leave-one-out is **not** applied here — it is train-only (Step B.12).


In [11]:

# ── Step B.13: 6-level fallback enrichment for df_valid and df_test ──
#
# All lookup maps are built from TRAIN aggregates.
# df_valid and df_test only READ from these maps — they never enter any aggregate.
#
# Fallback chain (new 6-level scheme):
#   Level 1 → degree_course_key               (exact same course in the same degree)
#   Level 2 → course_id                       (same course, any degree)
#   Level 3 → degree_id+req_type+credits      (similar courses in the same program)
#   Level 4 → faculty_id+req_type+credits     (similar courses in the same faculty)  ← NEW
#   Level 5 → req_type+credits                (similar courses, whole university)
#   Level 6 → global average                  (cold-start last resort)
#
# Two support columns:
#   course_history_count         → REAL course history (L1: sc_l1, L2: sc_l2, L3–6: 0)
#   difficulty_group_support_count → audit: raw support of the group that fired

# ── Pre-build lookup Series (O(1) per row via .map()) ──
_l1_pr = key_stats.set_index('degree_course_key')['course_pass_rate']
_l1_am = key_stats.set_index('degree_course_key')['course_avg_mark']
_l1_rr = key_stats.set_index('degree_course_key')['course_retake_rate']
_l1_sc = key_stats.set_index('degree_course_key')['support_count']

_l2_pr = cid_stats.set_index('course_id')['course_pass_rate']
_l2_am = cid_stats.set_index('course_id')['course_avg_mark']
_l2_rr = cid_stats.set_index('course_id')['course_retake_rate']
_l2_sc = cid_stats.set_index('course_id')['support_count']

_l3_pr = l3_stats.set_index('l3_key')['course_pass_rate']
_l3_am = l3_stats.set_index('l3_key')['course_avg_mark']
_l3_rr = l3_stats.set_index('l3_key')['course_retake_rate']
_l3_sc = l3_stats.set_index('l3_key')['support_count']

_l4_pr = l4_stats.set_index('l4_key')['course_pass_rate']
_l4_am = l4_stats.set_index('l4_key')['course_avg_mark']
_l4_rr = l4_stats.set_index('l4_key')['course_retake_rate']
_l4_sc = l4_stats.set_index('l4_key')['support_count']

_l5_pr = l5_stats.set_index('l5_key')['course_pass_rate']
_l5_am = l5_stats.set_index('l5_key')['course_avg_mark']
_l5_rr = l5_stats.set_index('l5_key')['course_retake_rate']
_l5_sc = l5_stats.set_index('l5_key')['support_count']


# ── Key-builder helpers ──

def _build_l3_key_for_split(df_split):
    """
    Level-3 key: "degree_id__requirement_type_id__credits_key"
    Returns NaN for rows where any required field is missing.
    We never build a key with a missing field — that would create
    bad strings like "nan__3__2".
    """
    all_present = (
        df_split['degree_id'].notna()
        & df_split['requirement_type_id'].notna()
        & df_split['course_credits'].notna()
    )
    result = pd.Series(index=df_split.index, dtype='object')
    rows_ok     = df_split.loc[all_present]
    credits_key = rows_ok['course_credits'].astype('float64').round().astype(int).astype(str)
    result.loc[all_present] = (
        rows_ok['degree_id'].astype(str)
        + '__'
        + rows_ok['requirement_type_id'].astype(str)
        + '__'
        + credits_key
    )
    return result


def _build_l4_key_for_split(df_split):
    """
    Level-4 key: "faculty_id__requirement_type_id__credits_key"
    Returns NaN for rows where faculty_id, requirement_type_id, or
    course_credits is missing. Missing faculty_id → skip Level 4.
    """
    all_present = (
        df_split['faculty_id'].notna()
        & df_split['requirement_type_id'].notna()
        & df_split['course_credits'].notna()
    )
    result = pd.Series(index=df_split.index, dtype='object')
    rows_ok     = df_split.loc[all_present]
    credits_key = rows_ok['course_credits'].astype('float64').round().astype(int).astype(str)
    result.loc[all_present] = (
        rows_ok['faculty_id'].astype(str)
        + '__'
        + rows_ok['requirement_type_id'].astype(str)
        + '__'
        + credits_key
    )
    return result


def _build_l5_key_for_split(df_split):
    """
    Level-5 key: "requirement_type_id__credits_key"
    Returns NaN for rows where requirement_type_id or course_credits is missing.
    """
    all_present = (
        df_split['requirement_type_id'].notna()
        & df_split['course_credits'].notna()
    )
    result = pd.Series(index=df_split.index, dtype='object')
    rows_ok     = df_split.loc[all_present]
    credits_key = rows_ok['course_credits'].astype('float64').round().astype(int).astype(str)
    result.loc[all_present] = (
        rows_ok['requirement_type_id'].astype(str)
        + '__'
        + credits_key
    )
    return result


def _enrich_split(df_split, split_name):
    """
    Apply the 6-level fallback chain to a validation or test split.

    Returns an enriched copy. The original df_split is not modified.
    No data from df_split enters any aggregate.

    course_history_count semantics:
      Level 1: support of the exact degree_course_key (real course history).
      Level 2: support of the course_id across all degrees (real course history).
      Levels 3-6: 0 — the estimate comes from a proxy group, NOT real history
                  for this specific course. Storing the group count here would
                  falsely inflate perceived confidence.

    difficulty_group_support_count:
      Raw support of whichever group actually provided the estimate.
      Kept for audit — tells you how many train observations backed the estimate,
      regardless of how specific or broad the key was.
      NOT included in model X features; inspect for debugging only.
    """

    out = df_split.copy()

    # ── Level 1 ──
    dck   = df_split['degree_course_key']
    pr_l1 = dck.map(_l1_pr)
    am_l1 = dck.map(_l1_am)
    rr_l1 = dck.map(_l1_rr)
    sc_l1 = dck.map(_l1_sc)
    l1_hit = pr_l1.notna()

    # ── Level 2 ──
    cid   = dck.str.rsplit('__', n=1).str[-1]
    pr_l2 = cid.map(_l2_pr)
    am_l2 = cid.map(_l2_am)
    rr_l2 = cid.map(_l2_rr)
    sc_l2 = cid.map(_l2_sc)
    l2_hit = (~l1_hit) & pr_l2.notna()

    # ── Level 3 ──
    l3_key = _build_l3_key_for_split(df_split)
    pr_l3  = l3_key.map(_l3_pr)
    am_l3  = l3_key.map(_l3_am)
    rr_l3  = l3_key.map(_l3_rr)
    sc_l3  = l3_key.map(_l3_sc)
    l3_hit = (~l1_hit) & (~l2_hit) & pr_l3.notna()

    # ── Level 4 (faculty) ──
    l4_key = _build_l4_key_for_split(df_split)
    pr_l4  = l4_key.map(_l4_pr)
    am_l4  = l4_key.map(_l4_am)
    rr_l4  = l4_key.map(_l4_rr)
    sc_l4  = l4_key.map(_l4_sc)
    l4_hit = (~l1_hit) & (~l2_hit) & (~l3_hit) & pr_l4.notna()

    # ── Level 5 (university-wide) ──
    l5_key = _build_l5_key_for_split(df_split)
    pr_l5  = l5_key.map(_l5_pr)
    am_l5  = l5_key.map(_l5_am)
    rr_l5  = l5_key.map(_l5_rr)
    sc_l5  = l5_key.map(_l5_sc)
    l5_hit = (~l1_hit) & (~l2_hit) & (~l3_hit) & (~l4_hit) & pr_l5.notna()

    # ── Level 6 (global) ──
    l6_hit = (~l1_hit) & (~l2_hit) & (~l3_hit) & (~l4_hit) & (~l5_hit)

    # ── Assign historical feature values ──
    out['course_pass_rate_historical'] = np.where(
        l1_hit.values, pr_l1.values,
        np.where(l2_hit.values, pr_l2.values,
        np.where(l3_hit.values, pr_l3.values,
        np.where(l4_hit.values, pr_l4.values,
        np.where(l5_hit.values, pr_l5.values,
                 global_vals['course_pass_rate'])))))

    out['course_avg_mark_historical'] = np.where(
        l1_hit.values, am_l1.values,
        np.where(l2_hit.values, am_l2.values,
        np.where(l3_hit.values, am_l3.values,
        np.where(l4_hit.values, am_l4.values,
        np.where(l5_hit.values, am_l5.values,
                 global_vals['course_avg_mark'])))))

    out['course_retake_rate_historical'] = np.where(
        l1_hit.values, rr_l1.values,
        np.where(l2_hit.values, rr_l2.values,
        np.where(l3_hit.values, rr_l3.values,
        np.where(l4_hit.values, rr_l4.values,
        np.where(l5_hit.values, rr_l5.values,
                 global_vals['course_retake_rate'])))))

    # course_history_count: REAL course history only.
    # Levels 3–6 get 0 — the group is not the same as the course.
    out['course_history_count'] = np.where(
        l1_hit.values, sc_l1.values,
        np.where(l2_hit.values, sc_l2.values,
                 0)).astype(int)

    # difficulty_group_support_count: raw support of the group that fired.
    # For audit only — not a model feature.
    out['difficulty_group_support_count'] = np.where(
        l1_hit.values, sc_l1.values,
        np.where(l2_hit.values, sc_l2.values,
        np.where(l3_hit.values, sc_l3.values,
        np.where(l4_hit.values, sc_l4.values,
        np.where(l5_hit.values, sc_l5.values,
                 global_support_count))))).astype(int)

    # difficulty_fallback_level: 1–6
    out['difficulty_fallback_level'] = np.where(
        l1_hit.values, 1,
        np.where(l2_hit.values, 2,
        np.where(l3_hit.values, 3,
        np.where(l4_hit.values, 4,
        np.where(l5_hit.values, 5,
                 6))))).astype(int)

    # course_difficulty_missing:
    #   0 ONLY for Level-1 with adequate support (confident exact estimate).
    #   Level 2 has real course_history_count but is still missing=1 (not an
    #   exact degree-course match — the degree pairing is new).
    #   Levels 3–6 are always missing=1.
    out['course_difficulty_missing'] = (
        (out['difficulty_fallback_level'] != 1)
        | (out['course_history_count'] < MIN_SUPPORT)
    ).astype(int)

    # ── Row-count and index-alignment guard ──
    assert len(out) == len(df_split), (
        f'FAIL ({split_name}): row count changed after enrichment. '
        f'Before: {len(df_split):,}  After: {len(out):,}'
    )
    assert out.index.equals(df_split.index), (
        f'FAIL ({split_name}): index changed after enrichment. '
        f'The join dropped, duplicated, or reordered rows.'
    )

    # ── Print fallback distribution ──
    n = len(out)
    fb_counts = out['difficulty_fallback_level'].value_counts().sort_index()
    print(f'\n{split_name} ({n:,} rows) — fallback distribution:')
    for lvl, cnt in fb_counts.items():
        print(f'  Level {int(lvl)}: {cnt:>8,} rows  ({cnt / n * 100:.2f}%)')

    return out


df_valid_enriched = _enrich_split(df_valid, 'valid')
df_test_enriched  = _enrich_split(df_test,  'test')

n_tr = len(df_train_enriched)
print(f'\ntrain ({n_tr:,} rows) — fallback distribution:')
tr_fb = df_train_enriched['difficulty_fallback_level'].value_counts().sort_index()
for lvl, cnt in tr_fb.items():
    print(f'  Level {int(lvl)}: {cnt:>8,} rows  ({cnt / n_tr * 100:.2f}%)')



valid (156,097 rows) — fallback distribution:
  Level 1:  120,858 rows  (77.42%)
  Level 2:    9,612 rows  (6.16%)
  Level 3:      619 rows  (0.40%)
  Level 4:    4,285 rows  (2.75%)
  Level 5:   20,723 rows  (13.28%)

test (110,008 rows) — fallback distribution:
  Level 1:   49,669 rows  (45.15%)
  Level 2:   25,640 rows  (23.31%)
  Level 3:      536 rows  (0.49%)
  Level 4:    8,442 rows  (7.67%)
  Level 5:   25,632 rows  (23.30%)
  Level 6:       89 rows  (0.08%)

train (450,465 rows) — fallback distribution:
  Level 1:  450,465 rows  (100.00%)


**What this shows:** How often each of the 6 fallback levels fires. Train is always 100% Level 1 by construction. For valid/test: Level 1 is the ideal case; Level 2 covers known courses in new degree pairings (~25,640 test rows); Levels 3–4 cover new course_ids with faculty or program-level signal; Levels 5–6 are true cold-start (very few rows expected after adding the faculty level). The diagnostic cell below provides the full 12-point audit including D8, which verifies that `course_history_count` is correctly zeroed for Levels 3–6 while `difficulty_group_support_count` retains the actual group support.


In [12]:

# ── Step B.13 Diagnostics: 12-point audit of the 6-level fallback scheme ──
#
# D1  faculty_id null percentage per split
# D2  Faculty-level (Level 4) coverage — among old-global rows
# D3  Fallback distribution summary across all three splits
# D4  old → new transition table (valid)
# D5  old → new transition table (test)
# D6  Where did old-global valid rows go in the new scheme?
# D7  Where did old-global test rows go in the new scheme?
# D8  Support-count distribution by level (BOTH columns, valid+test combined)
# D9  course_difficulty_missing distribution across all splits
# D10 Leakage proof: no valid/test course_id outside train cid_stats at Level 2
# D11 Leakage proof: train/valid/test index sets are disjoint
# D12 Key-casting sanity: no "nan" in any composite key

import warnings
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)

_level_labels = {
    1: 'exact degree_course_key        ',
    2: 'course_id cross-degree         ',
    3: 'degree+req_type+credits        ',
    4: 'faculty+req_type+credits (NEW) ',
    5: 'req_type+credits  (univ-wide)  ',
    6: 'global average (cold-start)    ',
}


# ════════════════════════════════════════════════════════════════════
# D1 — faculty_id null percentage per split
# ════════════════════════════════════════════════════════════════════
print('=' * 70)
print('D1 — faculty_id null percentage per split')
print('=' * 70)
for split_name, df_split in [('train', df_train), ('valid', df_valid), ('test', df_test)]:
    n_total = len(df_split)
    n_null  = int(df_split['faculty_id'].isna().sum())
    pct     = n_null / n_total * 100
    print(f'  {split_name:<6}: {n_null:>7,} / {n_total:>7,} null  ({pct:.4f}%)')


# ════════════════════════════════════════════════════════════════════
# D2 — Faculty-level (Level 4) coverage
# ════════════════════════════════════════════════════════════════════
print('\n' + '=' * 70)
print('D2 — Level-4 (faculty+req_type+credits) coverage per split')
print('     Among rows that reached Level 3+ (not Level 1 or 2)')
print('=' * 70)
for split_name, df_e in [('valid', df_valid_enriched), ('test', df_test_enriched)]:
    n_total     = len(df_e)
    n_l4        = int((df_e['difficulty_fallback_level'] == 4).sum())
    n_l3plus    = int((df_e['difficulty_fallback_level'] >= 3).sum())
    pct_total   = n_l4 / n_total * 100
    pct_coarse  = n_l4 / n_l3plus * 100 if n_l3plus > 0 else float('nan')
    print(f'  {split_name:<6}: Level-4 rows = {n_l4:>7,}  '
          f'({pct_total:.2f}% of total, {pct_coarse:.1f}% of L3+ rows)')


# ════════════════════════════════════════════════════════════════════
# D3 — Fallback distribution summary (all splits)
# ════════════════════════════════════════════════════════════════════
print('\n' + '=' * 70)
print('D3 — Fallback distribution summary (all splits)')
print('=' * 70)
for split_name, df_e in [
    ('train', df_train_enriched),
    ('valid', df_valid_enriched),
    ('test',  df_test_enriched),
]:
    n = len(df_e)
    print(f'\n  {split_name} ({n:,} rows):')
    fb = df_e['difficulty_fallback_level'].value_counts().sort_index()
    for lvl, cnt in fb.items():
        label = _level_labels.get(int(lvl), '?')
        print(f'    Level {int(lvl)} — {label}: {cnt:>8,}  ({cnt / n * 100:.2f}%)')


# ════════════════════════════════════════════════════════════════════
# D4 — old → new transition table (valid)
# ════════════════════════════════════════════════════════════════════
print('\n' + '=' * 70)
print('D4 — old → new transition table (valid split)')
print('     "old" = the scheme saved by the previous run of this notebook')
print('=' * 70)

if _old_col_valid is not None:
    _old_lv  = _old_col_valid.astype(int)
    _new_lv  = df_valid_enriched['difficulty_fallback_level'].astype(int)
    _old_max = int(_old_lv.max())
    print(f'  Old scheme had levels 1–{_old_max}')
    for old_lvl in sorted(_old_lv.unique()):
        mask_old   = _old_lv == old_lvl
        n_old      = int(mask_old.sum())
        new_counts = _new_lv[mask_old].value_counts().sort_index()
        print(f'\n  old Level {old_lvl}  ({n_old:,} rows) →')
        for new_lvl, cnt in new_counts.items():
            label = _level_labels.get(int(new_lvl), '?')
            print(f'      new Level {int(new_lvl)} — {label}: {cnt:>8,}  ({cnt / n_old * 100:.2f}%)')
else:
    print('  (no old level column — first run, no transition table)')


# ════════════════════════════════════════════════════════════════════
# D5 — old → new transition table (test)
# ════════════════════════════════════════════════════════════════════
print('\n' + '=' * 70)
print('D5 — old → new transition table (test split)')
print('=' * 70)

if _old_col_test is not None:
    _old_lv  = _old_col_test.astype(int)
    _new_lv  = df_test_enriched['difficulty_fallback_level'].astype(int)
    _old_max = int(_old_lv.max())
    print(f'  Old scheme had levels 1–{_old_max}')
    for old_lvl in sorted(_old_lv.unique()):
        mask_old   = _old_lv == old_lvl
        n_old      = int(mask_old.sum())
        new_counts = _new_lv[mask_old].value_counts().sort_index()
        print(f'\n  old Level {old_lvl}  ({n_old:,} rows) →')
        for new_lvl, cnt in new_counts.items():
            label = _level_labels.get(int(new_lvl), '?')
            print(f'      new Level {int(new_lvl)} — {label}: {cnt:>8,}  ({cnt / n_old * 100:.2f}%)')
else:
    print('  (no old level column — first run, no transition table)')


# ════════════════════════════════════════════════════════════════════
# D6 — Where did old-global valid rows go?
# ════════════════════════════════════════════════════════════════════
print('\n' + '=' * 70)
print('D6 — Old-global valid rows in the new scheme')
print('     "old-global" = the highest level number in the old scheme')
print('=' * 70)

if _old_col_valid is not None:
    _old_max_v          = int(_old_col_valid.max())
    _og_mask_v          = _old_col_valid == _old_max_v
    n_og_v              = int(_og_mask_v.sum())
    print(f'  Old max level = {_old_max_v}  →  {n_og_v:,} "old-global" rows in valid')
    if n_og_v > 0:
        new_sub = df_valid_enriched.loc[_og_mask_v, 'difficulty_fallback_level'].value_counts().sort_index()
        for new_lvl, cnt in new_sub.items():
            label = _level_labels.get(int(new_lvl), '?')
            print(f'    → new Level {int(new_lvl)} — {label}: {cnt:>8,}  ({cnt / n_og_v * 100:.2f}%)')
else:
    print('  (no old level column — first run)')


# ════════════════════════════════════════════════════════════════════
# D7 — Where did old-global test rows go?
# ════════════════════════════════════════════════════════════════════
print('\n' + '=' * 70)
print('D7 — Old-global test rows in the new scheme')
print('=' * 70)

if _old_col_test is not None:
    _old_max_t          = int(_old_col_test.max())
    _og_mask_t          = _old_col_test == _old_max_t
    n_og_t              = int(_og_mask_t.sum())
    print(f'  Old max level = {_old_max_t}  →  {n_og_t:,} "old-global" rows in test')
    if n_og_t > 0:
        new_sub = df_test_enriched.loc[_og_mask_t, 'difficulty_fallback_level'].value_counts().sort_index()
        for new_lvl, cnt in new_sub.items():
            label = _level_labels.get(int(new_lvl), '?')
            print(f'    → new Level {int(new_lvl)} — {label}: {cnt:>8,}  ({cnt / n_og_t * 100:.2f}%)')
else:
    print('  (no old level column — first run)')


# ════════════════════════════════════════════════════════════════════
# D8 — Support-count distribution by level (BOTH columns, valid+test)
# ════════════════════════════════════════════════════════════════════
print('\n' + '=' * 70)
print('D8 — Support-count distribution by level (valid + test combined)')
print()
print('  course_history_count           → should be 0 for Levels 3–6')
print('  difficulty_group_support_count → audit: raw support of the group that fired')
print('=' * 70)

_combined = pd.concat([df_valid_enriched, df_test_enriched], ignore_index=True)

for lvl in sorted(_combined['difficulty_fallback_level'].unique()):
    sub   = _combined[_combined['difficulty_fallback_level'] == lvl]
    label = _level_labels.get(int(lvl), '?')
    print(f'\n  Level {int(lvl)} — {label} ({len(sub):,} rows):')

    chc = sub['course_history_count']
    gsc = sub['difficulty_group_support_count']

    if int(lvl) >= 3:
        n_nonzero = int((chc != 0).sum())
        if n_nonzero > 0:
            print(f'    WARN: course_history_count has {n_nonzero:,} non-zero rows '
                  f'(expected 0 for Level {int(lvl)})')
        else:
            print(f'    course_history_count           : ALL ZERO  (correct for Level {int(lvl)})')
    else:
        print(f'    course_history_count           : '
              f'min={int(chc.min()):,}  median={int(chc.median()):,}  '
              f'max={int(chc.max()):,}')

    print(f'    difficulty_group_support_count : '
          f'min={int(gsc.min()):,}  median={int(gsc.median()):,}  '
          f'max={int(gsc.max()):,}')


# ════════════════════════════════════════════════════════════════════
# D9 — course_difficulty_missing distribution by split
# ════════════════════════════════════════════════════════════════════
print('\n' + '=' * 70)
print('D9 — course_difficulty_missing distribution by split')
print('     0 = confident  (Level 1 AND count >= MIN_SUPPORT)')
print('     1 = uncertain  (Levels 2–6, or Level-1 singleton)')
print('=' * 70)

for split_name, df_e in [
    ('train', df_train_enriched),
    ('valid', df_valid_enriched),
    ('test',  df_test_enriched),
]:
    n = len(df_e)
    print(f'\n  {split_name} ({n:,} rows):')
    cdm = df_e['course_difficulty_missing'].value_counts().sort_index()
    for val, cnt in cdm.items():
        print(f'    missing={int(val)}: {cnt:>8,}  ({cnt / n * 100:.2f}%)')

    conf_rows = df_e[df_e['course_difficulty_missing'] == 0]
    if len(conf_rows) > 0:
        bad_lvl = int((conf_rows['difficulty_fallback_level'] != 1).sum())
        bad_cnt = int((conf_rows['course_history_count'] < MIN_SUPPORT).sum())
        if bad_lvl > 0 or bad_cnt > 0:
            print(f'    WARN: {bad_lvl} missing=0 rows not Level-1, '
                  f'{bad_cnt} below MIN_SUPPORT — logic error!')
        else:
            print(f'    Check OK: all missing=0 rows are Level-1 with count >= {MIN_SUPPORT}')


# ════════════════════════════════════════════════════════════════════
# D10 — Leakage proof: valid/test Level-2 course_ids in train
# ════════════════════════════════════════════════════════════════════
print('\n' + '=' * 70)
print('D10 — Leakage proof: course_id at Level 2')
print('      All Level-2 rows must use course_ids present in train cid_stats')
print('=' * 70)

_train_cids = set(cid_stats['course_id'].astype(str))

for split_name, df_e in [('valid', df_valid_enriched), ('test', df_test_enriched)]:
    l2_mask = df_e['difficulty_fallback_level'] == 2
    l2_cids = set(
        df_e.loc[l2_mask, 'degree_course_key']
        .str.rsplit('__', n=1).str[-1].astype(str)
    )
    bad = l2_cids - _train_cids
    if len(bad) == 0:
        print(f'  {split_name:<6}: all {len(l2_cids):,} Level-2 course_ids are in cid_stats  ✓')
    else:
        print(f'  {split_name:<6}: FAIL — {len(bad):,} Level-2 course_ids NOT in cid_stats!')


# ════════════════════════════════════════════════════════════════════
# D11 — Leakage proof: index isolation
# ════════════════════════════════════════════════════════════════════
print('\n' + '=' * 70)
print('D11 — Leakage proof: index isolation (no row shared between splits)')
print('=' * 70)

_train_idx = set(df_train.index.tolist())
_valid_idx  = set(df_valid.index.tolist())
_test_idx   = set(df_test.index.tolist())

_v_overlap = _train_idx & _valid_idx
_t_overlap = _train_idx & _test_idx

if len(_v_overlap) == 0:
    print(f'  train ∩ valid index : 0  ✓')
else:
    print(f'  WARN: train ∩ valid index = {len(_v_overlap):,} rows — check split logic!')

if len(_t_overlap) == 0:
    print(f'  train ∩ test  index : 0  ✓')
else:
    print(f'  WARN: train ∩ test  index = {len(_t_overlap):,} rows — check split logic!')


# ════════════════════════════════════════════════════════════════════
# D12 — Key-casting sanity: no "nan" in composite keys
# ════════════════════════════════════════════════════════════════════
print('\n' + '=' * 70)
print('D12 — Key-casting sanity: check for "nan" in composite keys')
print('      Presence of "nan" means the null-mask in B.11 has a gap')
print('=' * 70)

_l3_bad = int(l3_stats['l3_key'].astype(str).str.contains('nan', case=False).sum())
_l4_bad = int(l4_stats['l4_key'].astype(str).str.contains('nan', case=False).sum())
_l5_bad = int(l5_stats['l5_key'].astype(str).str.contains('nan', case=False).sum())

print(f'  L3 keys containing "nan": {_l3_bad}  (should be 0)')
print(f'  L4 keys containing "nan": {_l4_bad}  (should be 0)')
print(f'  L5 keys containing "nan": {_l5_bad}  (should be 0)')

if _l3_bad + _l4_bad + _l5_bad == 0:
    print('  All composite keys are clean  ✓')
else:
    print('  FAIL: "nan" found in composite keys — check mask logic in B.11!')

print('\n' + '=' * 70)
print('All 12 diagnostics complete.')
print('=' * 70)


D1 — faculty_id null percentage per split
  train :       0 / 450,465 null  (0.0000%)
  valid :       0 / 156,097 null  (0.0000%)
  test  :       0 / 110,008 null  (0.0000%)

D2 — Level-4 (faculty+req_type+credits) coverage per split
     Among rows that reached Level 3+ (not Level 1 or 2)
  valid : Level-4 rows =   4,285  (2.75% of total, 16.7% of L3+ rows)
  test  : Level-4 rows =   8,442  (7.67% of total, 24.3% of L3+ rows)

D3 — Fallback distribution summary (all splits)

  train (450,465 rows):
    Level 1 — exact degree_course_key        :  450,465  (100.00%)

  valid (156,097 rows):
    Level 1 — exact degree_course_key        :  120,858  (77.42%)
    Level 2 — course_id cross-degree         :    9,612  (6.16%)
    Level 3 — degree+req_type+credits        :      619  (0.40%)
    Level 4 — faculty+req_type+credits (NEW) :    4,285  (2.75%)
    Level 5 — req_type+credits  (univ-wide)  :   20,723  (13.28%)

  test (110,008 rows):
    Level 1 — exact degree_course_key        :   49,

**Step B.14** — Sanity check: pick 10 `degree_course_key` values present in all three splits and confirm that `df_valid_enriched` and `df_test_enriched` show identical difficulty values for each (same train-lookup). Also show the `key_stats` reference values for comparison.

In [13]:

# ── Step B.14: Sanity check — identical values for keys present in all three splits ──
#
# For any degree_course_key present in train, valid, and test, the values in
# df_valid_enriched and df_test_enriched must be IDENTICAL (both read from the
# same key_stats table). The train row values will be CLOSE but not equal
# (LOO subtracts each row's own contribution from the aggregate).
#
# Cast all degree_course_key columns to str before set operations to avoid
# silent mismatches from dtype differences (e.g. category vs object).

keys_tr = set(df_train_enriched['degree_course_key'].astype(str).unique())
keys_vl = set(df_valid_enriched['degree_course_key'].astype(str).unique())
keys_ts = set(df_test_enriched['degree_course_key'].astype(str).unique())

keys_in_all = keys_tr & keys_vl & keys_ts

print(f'Keys present in all three splits : {len(keys_in_all):,}')

sample_keys = sorted(keys_in_all)[:10]

SHOW = [
    'degree_course_key',
    'course_history_count',
    'course_pass_rate_historical',
    'course_avg_mark_historical',
    'difficulty_fallback_level',
    'course_difficulty_missing',
]

pd.set_option('display.float_format', '{:.4f}'.format)

for split_name, df_e in [('valid', df_valid_enriched), ('test', df_test_enriched)]:
    print(f'\n--- {split_name} ---')
    # Cast key column to str before isin comparison to match sample_keys
    subset = (
        df_e.loc[df_e['degree_course_key'].astype(str).isin(sample_keys)]
        .drop_duplicates('degree_course_key')
        [SHOW]
        .sort_values('degree_course_key')
    )
    print(subset.to_string(index=False))

# key_stats reference for comparison (non-LOO values)
print('\n--- key_stats reference (non-LOO; train per-row values differ slightly by design) ---')
ref = (
    key_stats
    .loc[key_stats['degree_course_key'].astype(str).isin(sample_keys)]
    [['degree_course_key', 'support_count', 'course_pass_rate', 'course_avg_mark']]
    .sort_values('degree_course_key')
    .assign(difficulty_fallback_level=1)
)
print(ref.to_string(index=False))

pd.reset_option('display.float_format')


Keys present in all three splits : 700

--- valid ---
degree_course_key  course_history_count  course_pass_rate_historical  course_avg_mark_historical  difficulty_fallback_level  course_difficulty_missing
  1.111__1015.111                   192                       0.9001                     75.1257                          1                          0
  1.111__1016.111                   914                       0.9720                     79.5135                          1                          0
  1.111__1017.111                   299                       0.9211                     76.6729                          1                          0
  1.111__1018.111                   524                       0.8839                     66.7107                          1                          0
  1.111__1019.111                   938                       0.9330                     78.3190                          1                          0
  1.111__1020.111                  1119 

**What this shows:** For any given `degree_course_key`, the val and test rows should show **identical** `course_pass_rate`, `course_avg_mark`, and `support_count` (since they both look up from the same `key_stats` table). The `key_stats` reference row shows the non-LOO value, which is very close but not exactly equal to what individual train rows see (by design — the LOO version excludes each row's own contribution).

**Save** — Write the distinct **difficulty-generation** files `SPLIT_DATA_DIR/df_{train,valid,test}_difficulty.parquet` (D2). The base-generation inputs this notebook read from are never overwritten. The assertion verifies exactly the expected difficulty columns were added before saving.

In [14]:

# ── Save: write the distinct DIFFICULTY-generation split files (D2) ──
#
# This notebook owns df_*_difficulty.parquet ONLY. It never rewrites the base
# generation it read from (governance contracts 2-3).
#
# NEW_DIFF_COLS lists every column this notebook adds to the base splits.
# The assertion below computes the expected count from this list, so you
# never need to change a hardcoded number when columns are added or removed.
#
# Model-facing columns (included in X):
#   course_pass_rate_historical
#   course_avg_mark_historical
#   course_retake_rate_historical
#   course_history_count            <- REAL course history (0 for Levels 3-6)
#   difficulty_fallback_level
#   course_difficulty_missing
#
# Audit-only column (NOT in X):
#   difficulty_group_support_count  <- raw support of the group that fired

import os

DIFF_TRAIN_PATH = os.path.join(SPLIT_DATA_DIR, 'df_train_difficulty.parquet')
DIFF_VALID_PATH = os.path.join(SPLIT_DATA_DIR, 'df_valid_difficulty.parquet')
DIFF_TEST_PATH  = os.path.join(SPLIT_DATA_DIR, 'df_test_difficulty.parquet')

NEW_DIFF_COLS = [
    'course_pass_rate_historical',
    'course_avg_mark_historical',
    'course_retake_rate_historical',
    'course_history_count',
    'difficulty_fallback_level',
    'course_difficulty_missing',
    'difficulty_group_support_count',   # audit column - 7th
]

cols_before = df_train.shape[1]
cols_after  = df_train_enriched.shape[1]
n_new_cols  = cols_after - cols_before

print(f'Base column count (old difficulty cols already removed) : {cols_before}')
print(f'Column count after enrichment                          : {cols_after}')
print(f'New columns added                                      : {n_new_cols}  '
      f'(expected {len(NEW_DIFF_COLS)})')

assert n_new_cols == len(NEW_DIFF_COLS), (
    f'Expected exactly {len(NEW_DIFF_COLS)} new columns, got {n_new_cols}. '
    f'Expected list: {NEW_DIFF_COLS}'
)

# Verify every expected column is present in all three enriched splits
for col in NEW_DIFF_COLS:
    for label, df_e in [('train', df_train_enriched),
                         ('valid', df_valid_enriched),
                         ('test',  df_test_enriched)]:
        if col not in df_e.columns:
            raise AssertionError(f'Missing column "{col}" in {label} enriched split.')

print('New columns verified in all three splits:')
for c in NEW_DIFF_COLS:
    print(f'  {c}')

raise RuntimeError(
    'WRITE DISABLED: this legacy notebook was superseded by '
    'scripts/build_b2_temporal_course_stats.py. Use the versioned B2 builder.'
)

print()
for label, path, df_e in [
    ('train', DIFF_TRAIN_PATH, df_train_enriched),
    ('valid', DIFF_VALID_PATH, df_valid_enriched),
    ('test',  DIFF_TEST_PATH,  df_test_enriched),
]:
    size_mb = os.path.getsize(path) / 1_048_576
    print(f'  {label:<6}: {df_e.shape}  ->  {path}  ({size_mb:.1f} MB)')

Base column count (old difficulty cols already removed) : 63
Column count after enrichment                          : 70
New columns added                                      : 7  (expected 7)
New columns verified in all three splits:
  course_pass_rate_historical
  course_avg_mark_historical
  course_retake_rate_historical
  course_history_count
  difficulty_fallback_level
  course_difficulty_missing
  difficulty_group_support_count

  train : (450465, 70)  ->  D:\AI\Real projects\Academic_Advisor\data\model_data\df_train_difficulty.parquet  (18.8 MB)
  valid : (156097, 70)  ->  D:\AI\Real projects\Academic_Advisor\data\model_data\df_valid_difficulty.parquet  (6.4 MB)
  test  : (110008, 70)  ->  D:\AI\Real projects\Academic_Advisor\data\model_data\df_test_difficulty.parquet  (4.5 MB)


**What this shows:** The before/after column count confirms exactly 6 difficulty columns were added. The assertion fails loudly if there's a mismatch (e.g., a column was accidentally duplicated or omitted — the error message names the expected columns). After this cell runs, the files in `SPLIT_DATA_DIR` are the **final enriched splits** with the 5-level fallback scheme — downstream model notebooks should load from here. Note: `course_difficulty_missing = 0` means the model received a reliable Level-1 estimate; `= 1` means the estimate is coarser and the downstream model should treat it accordingly.